# 07 — Segment à faible taux de défaut (étape 6)
Segment défini **à l'octroi**, avant de regarder les défauts (`config/freddie_mac.yaml`, section `segment_faible_defaut`) : FICO ≥ 760, CLTV ≤ 70, DTI ≤ 36, millésime 2015.
On montre à partir de combien de défauts chaque test cesse d'être utilisable. Appellation : « segment à faible taux de défaut », pas « portefeuille à faibles défauts ».

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE / "src"))
from validation_toolkit.core.calibration_tests import binomial_exact, vasicek, intervalle_exact, puissance_binomial
from validation_toolkit.core.discrimination import gini

PROC = RACINE / "data" / "processed"

In [ ]:
AN = 2015
CIBLE = "defaut_hors_forbearance"
RHO = 0.15                      # hypothèse, à confirmer (voir notebook 05)
FRACTIONS = (1.0, 0.5, 0.25, 0.1, 0.05)
NB_TIRAGES = 300
GRAINE = 0

## 1. Le segment, et ses défauts sur les années de test

In [ ]:
panel = pd.read_pickle(PROC / f"panel_{AN}.pkl")
octroi = panel.drop_duplicates("id")
dans_segment = (octroi.fico >= 760) & (octroi.cltv <= 70) & (octroi.dti <= 36)
ids_seg = set(octroi.loc[dans_segment, "id"])
preds = pd.read_pickle(PROC / f"preds_{AN}_principal.pkl")
seg = preds[preds["id"].isin(ids_seg)]
print(f"{len(ids_seg):,} prêts à l'octroi ; {seg['id'].nunique():,} présents dans les années de test ; {len(seg):,} prêts-années")

def resume(df, nom):
    n, d = len(df), int(df[CIBLE].sum())
    lo, hi = intervalle_exact(n, d)
    out = {"population": nom, "prets_annees": n, "defauts": d, "taux_observe": d / n,
           "ic95_bas": lo, "ic95_haut": hi}
    for m in ("ref", "chall"):
        pr = df[f"pd_{m}"].mean()
        out[f"pd_{m}"] = pr
        out[f"p_binom_{m}"] = binomial_exact(n, d, pr, "sous")
        out[f"p_vasicek_{m}"] = vasicek(n, d, pr, RHO, "sous")
    return out

tab = pd.DataFrame([resume(seg, "segment"), resume(preds, "millésime entier")])
display(tab.round(4).T)

## 2. Ce que permet un nombre de défauts donné (calcul exact)
Pour une PD de référence `p0` (celle du segment) et un nombre de défauts **observés** `D` : largeur relative de l'intervalle exact (borne haute / borne basse) et puissance du test binomial à détecter une PD réelle 2 ou 3 fois plus grande que celle annoncée (n = D / p0).

In [ ]:
p0 = seg["pd_ref"].mean()
lignes = []
for D in (100, 50, 30, 20, 10, 5, 2, 1, 0):
    n = int(round(max(D, 0.5) / p0))
    lo, hi = intervalle_exact(n, D)
    lignes.append({"defauts_attendus": D, "n": n, "ic95_bas": lo, "ic95_haut": hi,
                   "haut_sur_bas": hi / lo if lo > 0 else np.inf,
                   "puissance_x2": puissance_binomial(n, p0, 2), "puissance_x3": puissance_binomial(n, p0, 3)})
puiss = pd.DataFrame(lignes)
print(f"PD de référence du segment : {p0:.4%}")
display(puiss.round(4))

## 3. Réduction progressive sur les données réelles
On tire au hasard une fraction des prêts du segment (sans remise, 300 tirages) : le nombre de défauts diminue. Pour chaque fraction : défauts moyens, part des tirages où le Gini est calculable, dispersion du Gini, largeur de l'intervalle exact.

In [ ]:
rng = np.random.default_rng(GRAINE)
ids = np.array(sorted(seg["id"].unique()))
par_id = {i: g for i, g in seg.groupby("id")}
lignes = []
for f in FRACTIONS:
    k = max(1, int(round(f * len(ids))))
    defauts, ginis, largeurs, calculables = [], [], [], 0
    for _ in range(NB_TIRAGES):
        choisis = rng.choice(ids, size=k, replace=False)
        d = seg[seg["id"].isin(choisis)]
        nd = int(d[CIBLE].sum())
        defauts.append(nd)
        lo, hi = intervalle_exact(len(d), nd)
        largeurs.append(hi - lo)
        if 0 < nd < len(d):
            calculables += 1
            ginis.append(gini(d[CIBLE], d["pd_ref"]))
    lignes.append({"fraction": f, "prets": k, "defauts_moyens": np.mean(defauts),
                   "part_gini_calculable": calculables / NB_TIRAGES,
                   "gini_moyen": np.mean(ginis) if ginis else np.nan,
                   "gini_ecart_type": np.std(ginis) if len(ginis) > 1 else np.nan,
                   "largeur_ic_taux": np.mean(largeurs)})
reduc = pd.DataFrame(lignes)
display(reduc.round(4))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogx(puiss.query("defauts_attendus > 0")["defauts_attendus"], puiss.query("defauts_attendus > 0")["puissance_x2"], "o-", label="PD réelle × 2")
ax[0].semilogx(puiss.query("defauts_attendus > 0")["defauts_attendus"], puiss.query("defauts_attendus > 0")["puissance_x3"], "o-", label="PD réelle × 3")
ax[0].axhline(0.8, color="0.6", ls="--"); ax[0].set_xlabel("défauts attendus"); ax[0].set_ylabel("puissance du test binomial"); ax[0].legend()
ax[1].errorbar(reduc["defauts_moyens"], reduc["gini_moyen"], yerr=reduc["gini_ecart_type"], fmt="o-")
ax[1].set_xscale("log"); ax[1].set_xlabel("défauts moyens dans le tirage"); ax[1].set_ylabel("Gini (± 1 écart-type)")
plt.tight_layout(); plt.show()

## Limites à retenir
- Les seuils du segment (760 / 70 / 36) sont un jugement, pas une règle réglementaire ; ils sont figés dans le YAML avant tout calcul de défauts.
- Les prêts-années d'un même prêt sont corrélés, et les années entre elles aussi : les intervalles exacts et les p-values du tableau 1 sont trop optimistes.
- Le test de Vasicek dépend de ρ (hypothèse non vérifiée ici).
- La puissance est calculée pour un test binomial sans corrélation : elle est donc plus élevée que celle d'un test corrigé.
- Le tirage de fractions fait baisser le nombre de défauts en conservant le même taux : il simule un portefeuille plus petit, pas un portefeuille plus sûr.
- À la fraction 1,0, l'écart-type du Gini est nul par construction (un seul tirage possible) : la précision du Gini à 15 défauts n'est pas mesurée par cette méthode, seulement sa dégradation en dessous.